# Phase 6: site/circuit metadata tables (`ami_site_metadata` / `ami_circuit_metadata`)

Fills a gap left open by Phase 5: none of `ami_raw`, `ami_meter`, or
`ami_raw_phaseseparate` carries site or circuit metadata (location,
capacity, manufacturer, DNSP, lifetime voltage/power-factor percentiles,
...) anywhere queryable alongside them in the local Parquet store. See
`lib/ami_metadata.py`'s module docstring for the full design rationale --
summarised here:

* **`ami_site_metadata`** -- one row per `site_id`. Genuine site-level
  attributes from `meta_up23c`, plus derived columns cheap to compute once
  here: `n_load_phases`/`n_pv_phases`, `s_99` (max-aggregated, matching
  `ami_build`'s own PV-normalization convention), `first_seen`/`last_seen`,
  and one presence flag per Phase 5 table (`in_ami_raw`, `in_ami_meter`,
  `in_ami_raw_phaseseparate`).
* **`ami_circuit_metadata`** -- one row per (`site_id`, `device_id`,
  `circuit_id`). Genuinely circuit-level attributes, mostly pass-through
  (including `n_long`/`n_lat`/`distance_km`, kept raw and unexamined --
  agreed 2026-09-04).

Both tables are scoped to whatever is **actually landed in the local
Parquet store right now** -- the union of `site_id`/`circuit_id` across the
three Phase 5 tables -- not to a specific historical notebook run's
in-memory resolution objects. That means this notebook is safe (and cheap)
to re-run any time the store changes (e.g. after widening `TRIAL_MONTHS` and
re-running `05_ami_build.ipynb`), without needing to replay Phase 4/5's
resolution notebook.

Four steps: (1) determine what's actually landed, (2) pull metadata from
Athena for exactly that scope, (3) build the two tables, (4) write them to
the store + save small artefacts.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents) if (p / "bms_sa_review").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import duckdb
import pandas as pd

from bms_sa_review.synthetic_ami_creation.config import ami_config as Config
from bms_sa_review.synthetic_ami_creation.lib import ami_athena as Athena
from bms_sa_review.synthetic_ami_creation.lib import ami_metadata as Meta


## 1. Determine what's actually landed

Read `site_id`/`circuit_id` directly from the local Parquet store's real
files -- not from any in-memory object from an earlier notebook run. `ami_raw`
has no `circuit_id` column (it's site-grain), so only `ami_meter` and
`ami_raw_phaseseparate` contribute to the circuit-level scope.


In [ ]:
SITE_LEVEL_TABLES = ["ami_raw", "ami_meter", "ami_raw_phaseseparate"]
CIRCUIT_LEVEL_TABLES = ["ami_meter", "ami_raw_phaseseparate"]

def _glob(table_name: str) -> str:
    return (Config.store_path(table_name) / "dt_month=*" / "*.parquet").as_posix()

SITE_TABLE_PATHS = {name: _glob(name) for name in SITE_LEVEL_TABLES}
CIRCUIT_TABLE_PATHS = {name: _glob(name) for name in CIRCUIT_LEVEL_TABLES}

con = duckdb.connect()
PRESENCE = Meta.determine_landed_scope(con, SITE_TABLE_PATHS)
for name, ids in PRESENCE.items():
    print(f"{name}: {len(ids):,} distinct site_id")


In [ ]:
ALL_SITE_IDS = sorted(set().union(*PRESENCE.values())) if PRESENCE else []
ALL_CIRCUIT_IDS = sorted(Meta.determine_landed_circuit_scope(con, CIRCUIT_TABLE_PATHS))

print(f"{len(ALL_SITE_IDS):,} distinct site_id across the union of {SITE_LEVEL_TABLES}.")
print(f"{len(ALL_CIRCUIT_IDS):,} distinct circuit_id across the union of {CIRCUIT_LEVEL_TABLES}.")


## 2. Pull metadata from Athena, scoped to exactly what's landed

Chunked queries against `meta_up23c` (small, unpartitioned -- no partition
predicate required). This is the only Athena-touching step in this
notebook; everything after this cell is local.


In [ ]:
site_meta_raw = Meta.pull_site_level_meta(Athena.aq, ALL_SITE_IDS, database=Config.SAI)
print(f"site_meta_raw: {len(site_meta_raw):,} rows (expected {len(ALL_SITE_IDS):,} sites).")
Athena.scan_report()


In [ ]:
circuit_meta_raw = Meta.pull_circuit_level_meta(Athena.aq, ALL_CIRCUIT_IDS, database=Config.SAI)
print(f"circuit_meta_raw: {len(circuit_meta_raw):,} rows (expected {len(ALL_CIRCUIT_IDS):,} circuits).")
Athena.scan_report()


## 3. Build the two tables


In [ ]:
ami_site_metadata = Meta.build_ami_site_metadata(site_meta_raw, circuit_meta_raw, PRESENCE)
ami_circuit_metadata = Meta.build_ami_circuit_metadata(circuit_meta_raw)

print(f"ami_site_metadata: {len(ami_site_metadata):,} rows, {ami_site_metadata.shape[1]} columns")
print(f"ami_circuit_metadata: {len(ami_circuit_metadata):,} rows, {ami_circuit_metadata.shape[1]} columns")
display(ami_site_metadata.head())
display(ami_circuit_metadata.head())


In [ ]:
missing_site_ids = sorted(set(ALL_SITE_IDS) - set(site_meta_raw.site_id))
print(f"{len(missing_site_ids):,} landed site_id(s) with NO matching row in meta_up23c "
      "(would show up with null site-level columns above).")
if missing_site_ids:
    print(missing_site_ids[:20], "..." if len(missing_site_ids) > 20 else "")


## 4. Write to the local store + save small artefacts

Single (non-partitioned) Parquet files, per `write_metadata_table`'s
docstring -- both tables are small enough (thousands to tens of thousands
of rows) that Hive-partitioning would just fragment the file.


In [ ]:
site_metadata_path = Meta.write_metadata_table(ami_site_metadata, Config.store_path("ami_site_metadata"))
circuit_metadata_path = Meta.write_metadata_table(ami_circuit_metadata, Config.store_path("ami_circuit_metadata"))
print(f"Wrote {site_metadata_path}")
print(f"Wrote {circuit_metadata_path}")


## 5. Summary


In [ ]:
summary_rows = []
for name in SITE_LEVEL_TABLES:
    summary_rows.append({"table": name, "n_sites": len(PRESENCE.get(name, set()))})
summary = pd.DataFrame(summary_rows)
display(summary)

print(f"\nami_site_metadata: {len(ami_site_metadata):,} sites, "
      f"{len(missing_site_ids):,} with no meta_up23c match.")
print(f"ami_circuit_metadata: {len(ami_circuit_metadata):,} circuits.")
